# embedding test (eager, pydantic ai)

> proof of concept: embeds a small sample of the chunks produced by `3.chunking.ipynb` (`new_chunks.parquet`).
>
> embedding is async network I/O, so it runs eagerly on a collected dataframe - notebooks support top-level await
>
> requires `OPENAI_API_KEY` (and optionally `OPENAI_BASE_URL`) environment variables

In [ ]:
import polars as pl

In [ ]:
%env OPENAI_API_KEY=sk-xxxxxxxxxxxxxxxxxxxx
%env OPENAI_BASE_URL=https://api.ai.tech.gov.sg/platform/models

In [ ]:
# small sample of the chunks built in 3.chunking.ipynb
sample_chunks = pl.read_parquet("new_chunks.parquet").head(20)
sample_chunks

In [ ]:
import os

from pydantic_ai import Embedder
from pydantic_ai.embeddings import EmbeddingSettings
from pydantic_ai.embeddings.openai import OpenAIEmbeddingModel
from pydantic_ai.providers.openai import OpenAIProvider

EMBEDDING_MODEL = "text-embedding-3-small"
EMBEDDING_DIM = 1536

# same embedder setup as preprocess.py (create_embedder)
# base_url must include the scheme, otherwise the openai client only
# fails at request time with UnsupportedProtocol / "Connection error"
base_url = (os.environ.get("OPENAI_BASE_URL") or "").strip() or None
if base_url and not base_url.startswith(("http://", "https://")):
    base_url = "https://" + base_url

provider = OpenAIProvider(
    api_key=os.environ["OPENAI_API_KEY"],
    base_url=base_url,
)
model = OpenAIEmbeddingModel(EMBEDDING_MODEL, provider=provider)
embedder = Embedder(model, settings=EmbeddingSettings(dimensions=EMBEDDING_DIM))

In [ ]:
BATCH_SIZE = 4

texts = sample_chunks.get_column("retrieval_text").to_list()
total_batches = (len(texts) + BATCH_SIZE - 1) // BATCH_SIZE

embeddings: list[list[float]] = []
for i in range(0, len(texts), BATCH_SIZE):
    batch = texts[i : i + BATCH_SIZE]
    embeddings.extend(await embedder.embed_documents(batch))

    print(f"batch {i // BATCH_SIZE + 1}/{total_batches} done, "
        f"processed {min(i + BATCH_SIZE, len(texts))}/{len(texts)}")

In [ ]:
# attach embeddings as a list[f32] column and sanity check
assert len(embeddings) == len(sample_chunks), "one embedding per chunk"
assert all(len(vec) == EMBEDDING_DIM for vec in embeddings), f"all embeddings must be {EMBEDDING_DIM}-dim"

sample_chunks = sample_chunks.with_columns(
    embedding=pl.Series("embedding", embeddings, dtype=pl.List(pl.Float32))
)

sample_chunks

# Notes for the full run

- embed every `retrieval_text` (drop the `.head(20)`), attach the column, then `write_parquet("new_chunks_embedded.parquet")`
- the production version with rate-limit pacing, retries and checkpoint/resume lives in `pipeline/phase4_embed.py`